# Laboratorio 5: clasificación multiclase Quick, Draw! con PyTorch

**Autor:** Nataniel Mauricio Arapa Estrada  
**Materia:** SIS420 — Inteligencia Artificial I

Se clasifican dibujos de cuatro categorías: **manzana, auto, reloj y gato**. Para conservar el objetivo del cuadernillo original se usa regresión logística multiclase, implementada como `nn.Linear(784, 4)` y entrenada con `CrossEntropyLoss`.

Todo el aprendizaje, división estratificada, lotes, inferencia y evaluación utiliza PyTorch. NumPy aparece únicamente para abrir los archivos externos `.npy`, un formato cuyo decodificador no forma parte de PyTorch; los datos se convierten inmediatamente a tensores.


## 1. Configuración


In [ ]:
from pathlib import Path
import random
import shutil
import subprocess
import sys
import urllib.request

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

print(f"PyTorch {torch.__version__} | Dispositivo: {DEVICE}")


## 2. Localización de los datos

La celda busca cada `.npy` en el repositorio y en las rutas originales de Drive. Si solo encuentra los `.rar`, intenta extraerlos con `unrar`. Como último recurso descarga los archivos oficiales de Quick, Draw!.

La descarga completa puede superar 300 MB. Si ya tienes los `.npy`, colócalos en `Lab5`, junto al notebook o en la carpeta de datasets de Drive para evitarla.


In [ ]:
CLASES = ["apple", "car", "clock", "cat"]
NOMBRES_ES = ["Manzana", "Auto", "Reloj", "Gato"]
DESCARGAR_SI_FALTAN = True

BASES = [
    Path.cwd(),
    Path.cwd() / "Lab5",
    Path.cwd().parent / "Lab5",
    Path("/content/drive/MyDrive/Colab Notebooks/machine learning/datasets"),
    Path("/content/gdrive/MyDrive/Colab Notebooks/machine learning/datasets"),
]


def localizar_npy(nombre):
    for base in BASES:
        if not base.exists():
            continue
        candidatos = [
            base / f"{nombre}.npy",
            base / f"full_numpy_bitmap_{nombre}.npy",
        ]
        candidatos.extend(base.glob(f"*{nombre}*.npy"))
        for candidato in candidatos:
            if candidato.exists():
                return candidato
    return None


def intentar_extraer_rar(nombre):
    ejecutable = shutil.which("unrar")
    if ejecutable is None:
        return None
    for base in BASES:
        archivo_rar = base / f"{nombre}.rar"
        if archivo_rar.exists():
            print(f"Extrayendo {archivo_rar.name}...")
            subprocess.run(
                [ejecutable, "e", "-o+", str(archivo_rar), str(base)],
                check=True,
                stdout=subprocess.DEVNULL,
            )
            return localizar_npy(nombre)
    return None


def descargar_npy(nombre):
    destino_dir = Path.cwd() / "quickdraw_npy"
    destino_dir.mkdir(exist_ok=True)
    destino = destino_dir / f"{nombre}.npy"
    url = (
        "https://storage.googleapis.com/quickdraw_dataset/full/"
        f"numpy_bitmap/{nombre}.npy"
    )
    print(f"Descargando {nombre}.npy...")
    urllib.request.urlretrieve(url, destino)
    return destino


rutas_npy = {}
for clase in CLASES:
    ruta = localizar_npy(clase) or intentar_extraer_rar(clase)
    if ruta is None and DESCARGAR_SI_FALTAN:
        ruta = descargar_npy(clase)
    if ruta is None:
        raise FileNotFoundError(
            f"Falta {clase}.npy. Extrae {clase}.rar o activa la descarga."
        )
    rutas_npy[clase] = ruta

print("Archivos listos:")
for clase, ruta in rutas_npy.items():
    print(f"  {clase:>5}: {ruta}")


## 3. Construcción balanceada del dataset

Se toman 15 000 ejemplos por clase. Los píxeles permanecen como `uint8` en memoria para no cuadruplicar el consumo; cada lote se convierte a `float32` y se divide entre 255 justo antes de entrar al modelo.


In [ ]:
EJEMPLOS_POR_CLASE = 15_000
tensores_x, tensores_y = [], []

for etiqueta, clase in enumerate(CLASES):
    arreglo = np.load(rutas_npy[clase], mmap_mode="r")
    if len(arreglo) < EJEMPLOS_POR_CLASE:
        raise ValueError(
            f"{clase}.npy tiene {len(arreglo)} ejemplos; se requieren {EJEMPLOS_POR_CLASE}."
        )
    # NumPy solo decodifica .npy; a partir de aquí todo es un tensor.
    x_clase = torch.from_numpy(
        np.array(arreglo[:EJEMPLOS_POR_CLASE], dtype=np.uint8, copy=True)
    )
    y_clase = torch.full((EJEMPLOS_POR_CLASE,), etiqueta, dtype=torch.long)
    tensores_x.append(x_clase)
    tensores_y.append(y_clase)

X = torch.cat(tensores_x, dim=0)
y = torch.cat(tensores_y, dim=0)
del tensores_x, tensores_y

print(f"X: {X.shape}, {X.dtype}")
print(f"y: {y.shape}, {y.dtype}")
print(f"Conteos: {torch.bincount(y).tolist()}")


In [ ]:
fig, ejes = plt.subplots(4, 3, figsize=(8, 10))
for clase in range(4):
    indices = torch.where(y == clase)[0][:3]
    for columna, indice in enumerate(indices):
        ejes[clase, columna].imshow(X[indice].reshape(28, 28).tolist(), cmap="gray")
        ejes[clase, columna].set_title(NOMBRES_ES[clase])
        ejes[clase, columna].axis("off")
plt.tight_layout()
plt.show()


## 4. División estratificada y DataLoaders


In [ ]:
def indices_estratificados(y, proporcion_prueba=0.20, semilla=SEED):
    generador = torch.Generator().manual_seed(semilla)
    train_partes, test_partes = [], []
    for clase in torch.unique(y):
        indices = torch.where(y == clase)[0]
        indices = indices[torch.randperm(len(indices), generator=generador)]
        n_test = int(len(indices) * proporcion_prueba)
        test_partes.append(indices[:n_test])
        train_partes.append(indices[n_test:])

    idx_train = torch.cat(train_partes)
    idx_test = torch.cat(test_partes)
    idx_train = idx_train[torch.randperm(len(idx_train), generator=generador)]
    idx_test = idx_test[torch.randperm(len(idx_test), generator=generador)]
    return idx_train, idx_test


idx_train, idx_test = indices_estratificados(y)
X_train, y_train = X[idx_train], y[idx_train]
X_test, y_test = X[idx_test], y[idx_test]
del X, y

generador_loader = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=512,
    shuffle=True,
    generator=generador_loader,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    TensorDataset(X_test, y_test),
    batch_size=1024,
    shuffle=False,
    pin_memory=torch.cuda.is_available(),
)

print(f"Entrenamiento: {len(X_train):,} | Prueba: {len(X_test):,}")
print(f"Conteos test: {torch.bincount(y_test).tolist()}")


## 5. Regresión logística multiclase

La capa lineal produce cuatro logits. `CrossEntropyLoss` aplica internamente `log_softmax` y la pérdida de entropía cruzada, por lo que durante el entrenamiento **no** se añade una sigmoide ni una softmax manual.


In [ ]:
class RegresionLogisticaMulticlase(nn.Module):
    def __init__(self, entradas=28 * 28, clases=4):
        super().__init__()
        self.lineal = nn.Linear(entradas, clases)

    def forward(self, x):
        return self.lineal(x)


def preparar_pixeles(x):
    return x.to(DEVICE, dtype=torch.float32, non_blocking=True) / 255.0


@torch.inference_mode()
def evaluar(modelo, loader, criterio):
    modelo.eval()
    perdida_total, aciertos, total = 0.0, 0, 0
    predicciones, reales = [], []
    for xb, yb in loader:
        xb = preparar_pixeles(xb)
        yb = yb.to(DEVICE, non_blocking=True)
        logits = modelo(xb)
        perdida = criterio(logits, yb)
        pred = logits.argmax(dim=1)
        perdida_total += perdida.item() * len(yb)
        aciertos += (pred == yb).sum().item()
        total += len(yb)
        predicciones.append(pred.cpu())
        reales.append(yb.cpu())
    return (
        perdida_total / total,
        aciertos / total,
        torch.cat(predicciones),
        torch.cat(reales),
    )


torch.manual_seed(SEED)
modelo = RegresionLogisticaMulticlase().to(DEVICE)
criterio = nn.CrossEntropyLoss()
optimizador = torch.optim.AdamW(modelo.parameters(), lr=0.003, weight_decay=1e-4)


In [ ]:
EPOCAS = 25
historial_train, historial_test, historial_accuracy = [], [], []

for epoca in range(EPOCAS):
    modelo.train()
    perdida_acumulada = 0.0
    for xb, yb in train_loader:
        xb = preparar_pixeles(xb)
        yb = yb.to(DEVICE, non_blocking=True)

        optimizador.zero_grad()
        logits = modelo(xb)
        perdida = criterio(logits, yb)
        perdida.backward()
        optimizador.step()
        perdida_acumulada += perdida.item() * len(yb)

    train_loss = perdida_acumulada / len(X_train)
    test_loss, test_acc, _, _ = evaluar(modelo, test_loader, criterio)
    historial_train.append(train_loss)
    historial_test.append(test_loss)
    historial_accuracy.append(test_acc)

    print(
        f"Época {epoca + 1:>2}/{EPOCAS} | "
        f"train CE: {train_loss:.4f} | test CE: {test_loss:.4f} | "
        f"accuracy: {100 * test_acc:.2f}%"
    )


## 6. Evaluación por clase


In [ ]:
perdida_test, accuracy_test, pred_test, real_test = evaluar(
    modelo, test_loader, criterio
)
matriz = torch.zeros((4, 4), dtype=torch.long)
for real, pred in zip(real_test, pred_test):
    matriz[real, pred] += 1

exactitud_clase = matriz.diag() / matriz.sum(dim=1).clamp_min(1)
print(f"Cross-entropy de prueba: {perdida_test:.4f}")
print(f"Accuracy total: {100 * accuracy_test:.2f}%")
print("\nMatriz de confusión (filas=reales, columnas=predichas):")
print(matriz)
print("\nAccuracy por clase:")
for nombre, valor in zip(NOMBRES_ES, exactitud_clase):
    print(f"{nombre:>8}: {100 * valor.item():.2f}%")


In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(13, 4.5))
ejes[0].plot(historial_train, label="Entrenamiento")
ejes[0].plot(historial_test, label="Prueba")
ejes[0].set(title="Entropía cruzada", xlabel="Época", ylabel="CE")
ejes[0].legend()
ejes[0].grid(alpha=0.3)

ejes[1].bar(NOMBRES_ES, (100 * exactitud_clase).tolist())
ejes[1].set(title="Accuracy por clase", ylabel="Accuracy (%)", ylim=(0, 100))
ejes[1].grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## 7. Inferencia visual


In [ ]:
generador_demo = torch.Generator().manual_seed(SEED + 1)
indices_demo = torch.randperm(len(X_test), generator=generador_demo)[:8]
imagenes_demo = X_test[indices_demo]
reales_demo = y_test[indices_demo]

modelo.eval()
with torch.inference_mode():
    logits_demo = modelo(preparar_pixeles(imagenes_demo))
    probabilidades_demo = torch.softmax(logits_demo, dim=1).cpu()
    pred_demo = probabilidades_demo.argmax(dim=1)

fig, ejes = plt.subplots(2, 4, figsize=(11, 6))
for eje, imagen, real, pred, probs in zip(
    ejes.flat, imagenes_demo, reales_demo, pred_demo, probabilidades_demo
):
    eje.imshow(imagen.reshape(28, 28).tolist(), cmap="gray")
    color = "green" if pred == real else "crimson"
    eje.set_title(
        f"Pred: {NOMBRES_ES[pred]} ({100 * probs[pred]:.1f}%)\n"
        f"Real: {NOMBRES_ES[real]}",
        color=color,
    )
    eje.axis("off")
plt.tight_layout()
plt.show()


## Conclusiones

- Una única capa lineal reproduce la regresión logística multiclase del laboratorio sin SciPy ni el esquema manual one-vs-all.
- `CrossEntropyLoss` aprende las cuatro clases simultáneamente y es más estable que calcular logaritmos de probabilidades a mano.
- La matriz de confusión y la exactitud por clase permiten detectar clases particularmente difíciles, incluso cuando el accuracy global parece bueno.
- Si se quisiera superar el límite de un clasificador lineal, el paso siguiente sería reemplazarlo por una CNN; no se hace aquí para conservar el tipo de modelo solicitado en el laboratorio original.
